# Stage 0 Drive audit

Mount Google Drive, ensure the required Petukhov Text-to-Visualization root exists, and print a shallow tree for the Stage 0 review.

In [ ]:
from __future__ import annotations

import json
import os
import platform
from datetime import datetime, timezone
from pathlib import Path

from google.colab import drive

mount_error = None
try:
    drive.mount('/content/drive', force_remount=False)
except Exception as exc:
    mount_error = repr(exc)
    if not Path('/content/drive/MyDrive').exists():
        print('STAGE0_DRIVE_AUDIT_BLOCKED')
        print(json.dumps({
            'status': 'blocked',
            'reason': 'drive.mount failed and /content/drive/MyDrive is not available',
            'mount_error': mount_error,
            'hint': 'Grant Google Drive access in the active Colab session, then rerun this cell through the runner.',
        }, ensure_ascii=False, indent=2))
        raise
    print('[warn] drive.mount failed, but /content/drive/MyDrive already exists; continuing with existing mount')

drive_root = Path('/content/drive/MyDrive/petr_text_to_visualization_part')
legacy_diploma_root = Path('/content/drive/MyDrive/diploma/petr_text_to_visualization_part')
drive_root.mkdir(parents=True, exist_ok=True)

def shallow_tree(path: Path, max_depth: int = 3, max_items_per_dir: int = 50):
    rows = []
    base_depth = len(path.parts)
    if not path.exists():
        return rows
    stack = [path]
    while stack:
        current = stack.pop(0)
        depth = len(current.parts) - base_depth
        if depth > max_depth:
            continue
        try:
            stat = current.stat()
        except OSError:
            stat = None
        rows.append({
            'path': str(current.relative_to(path)) if current != path else '.',
            'kind': 'dir' if current.is_dir() else 'file',
            'size_bytes': None if current.is_dir() or stat is None else stat.st_size,
        })
        if current.is_dir() and depth < max_depth:
            try:
                children = sorted(current.iterdir(), key=lambda p: (not p.is_dir(), p.name.lower()))[:max_items_per_dir]
            except OSError:
                children = []
            stack.extend(children)
    return rows

audit = {
    'status': 'ok',
    'created_or_exists': drive_root.exists(),
    'mount_error': mount_error,
    'drive_root': str(drive_root),
    'legacy_diploma_root': str(legacy_diploma_root),
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'python': platform.python_version(),
    'platform': platform.platform(),
    'cwd': os.getcwd(),
    'tree': shallow_tree(drive_root),
    'legacy_diploma_tree': shallow_tree(legacy_diploma_root) if legacy_diploma_root.exists() else [],
}

print('STAGE0_DRIVE_AUDIT_OK')
print(json.dumps(audit, ensure_ascii=False, indent=2))
